In [ ]:
# ==========================================================
# BIBLIOTECAS
# ==========================================================

import json
import re

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
# ==========================================================
# CONFIGURAÇÃO
# ==========================================================


def resolver_base() -> Path:
    """Resolve o diretório raiz do projeto em qualquer ambiente."""
    cwd = Path.cwd().resolve()

    for candidate in [cwd, *cwd.parents]:
        if (candidate / "tsv").exists() and (candidate / "catalogos").exists():
            return candidate

    return cwd


BASE = resolver_base()

DIR_TSV = BASE / "tsv"
DIR_CATALOGOS = BASE / "catalogos"
DIR_SAIDA = BASE / "normalizado"

if not DIR_CATALOGOS.exists():
    raise FileNotFoundError(f"Diretório de catálogos não encontrado: {DIR_CATALOGOS}")

DIR_SAIDA.mkdir(
    parents=True,
    exist_ok=True
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# ==========================================================
# CATÁLOGOS
# ==========================================================

def carregar_json(arquivo):
    with open(arquivo, encoding="utf-8") as f:
        return json.load(f)


INDICE_CURSOS = carregar_json(
    DIR_CATALOGOS / "indice_cursos.json"
)

CATALOGO_CURSOS = carregar_json(
    DIR_CATALOGOS / "catalogo_cursos.json"
)

# ==========================================================
# CATÁLOGO DOS CAMPI
# ==========================================================

INDICE_CAMPI = {

    "Campus Darcy Ribeiro":
        "Darcy Ribeiro",

    "Darcy Ribeiro":
        "Darcy Ribeiro",

    "Darcy Ribeiro (Plano Piloto) / DF":
        "Darcy Ribeiro",

    "Campus UnB Ceilândi a (FCE)":
        "Ceilândia",

    "Campus UnB Ceilândia (FCE)":
        "Ceilândia",

    "Ceilândia / DF":
        "Ceilândia",

    "Ceilândi a (FCE)":
        "Ceilândia",

    "Gama (FGA)":
        "Gama",

    "Campus UnB Gama (FGA)":
        "Gama",

    "Gama / DF":
        "Gama",

    "Campus UnB Planalti na (FUP)":
        "Planaltina",

    "Campus UnB Planaltina (FUP)":
        "Planaltina",

    "Planalti na (FUP)":
        "Planaltina",

    "Planaltina / DF":
        "Planaltina"

}

In [9]:
# ==========================================================
# ESTRUTURA DO PAS
# ==========================================================

CAMPUS_LINHA = 1
CAMPUS_COLUNA = 2

# ==========================================================
# MODALIDADES
# ==========================================================

MODALIDADES = {

    # ------------------------------------------------------
    # AMPLA CONCORRÊNCIA
    # ------------------------------------------------------

    "AC": {
        "modalidade_normalizada": "AC",
        "escola_publica": False,
        "faixa_renda": None,
        "grupo_etnico": None,
        "pcd": False
    },

    # ------------------------------------------------------
    # COTA PARA NEGROS
    # ------------------------------------------------------

    "CN": {
        "modalidade_normalizada": "CN",
        "escola_publica": False,
        "faixa_renda": None,
        "grupo_etnico": "NEGROS",
        "pcd": False
    },

    # ------------------------------------------------------
    # ESCOLA PÚBLICA
    # ------------------------------------------------------

    **{
        modalidade: {
            "modalidade_normalizada": modalidade,
            "escola_publica": True,
            "faixa_renda": modalidade.split("_")[1],
            "grupo_etnico": modalidade.split("_")[2],
            "pcd": modalidade.endswith("PCD")
        }

        for modalidade in (

            "EP_R1_PPI",
            "EP_R1_PPI_PCD",
            "EP_R1_NPPI",
            "EP_R1_NPPI_PCD",

            "EP_R2_PPI",
            "EP_R2_PPI_PCD",
            "EP_R2_NPPI",
            "EP_R2_NPPI_PCD",

            "EP_R1_PPIQ",
            "EP_R1_PPIQ_PCD",
            "EP_R1_NPPIQ",
            "EP_R1_NPPIQ_PCD",

            "EP_R2_PPIQ",
            "EP_R2_PPIQ_PCD",
            "EP_R2_NPPIQ",
            "EP_R2_NPPIQ_PCD",

        )
    }

}

# ==========================================================
# CONFIGURAÇÃO DOS TRIÊNIOS
# ==========================================================

CONFIG_TRIENIOS = {

    "2018-2020": {
        "ano_ingresso": 2020,
        "limite_r1": 1.5
    },

    "2019-2021": {
        "ano_ingresso": 2021,
        "limite_r1": 1.5
    },

    "2020-2022": {
        "ano_ingresso": 2022,
        "limite_r1": 1.5
    },

    "2021-2023": {
        "ano_ingresso": 2023,
        "limite_r1": 1.5
    },

    "2022-2024": {
        "ano_ingresso": 2024,
        "limite_r1": 1.5
    },

    "2023-2025": {
        "ano_ingresso": 2025,
        "limite_r1": 1.0
    }

}

# ==========================================================
# LAYOUT DOS BOLETINS
# ==========================================================

LAYOUTS = {

    "2018-2020": {
        "header": 4,
        "tipo": CAMPUS_LINHA,
        "curso_col": 0,
        "turno_col": 1,
        "campus_col": None,
        "campus_prefixo": "Campus UnB —",
        "modalidades": {
            (2,3):"EP_R1_PPI",
            (4,5):"EP_R1_PPI_PCD",
            (6,7):"EP_R1_NPPI",
            (8,9):"EP_R1_NPPI_PCD",
            (10,11):"EP_R2_PPI",
            (12,13):"EP_R2_PPI_PCD",
            (14,15):"EP_R2_NPPI",
            (16,17):"EP_R2_NPPI_PCD",
            (18,19):"CN",
            (20,21):"AC"
        }
    },

    "2019-2021": {
        "header": 4,
        "tipo": CAMPUS_COLUNA,
        "campus_col": 0,
        "turno_col": 1,
        "curso_col": 2,
        "campus_prefixo": None,
        "modalidades": {
            (3,4):"CN",
            (5,6):"EP_R1_PPI",
            (7,8):"EP_R1_PPI_PCD",
            (9,10):"EP_R1_NPPI",
            (11,12):"EP_R1_NPPI_PCD",
            (13,14):"EP_R2_PPI",
            (15,16):"EP_R2_PPI_PCD",
            (17,18):"EP_R2_NPPI",
            (19,20):"EP_R2_NPPI_PCD",
            (21,22):"AC"
        }
    },

    "2020-2022": {
        "header": 4,
        "tipo": CAMPUS_LINHA,
        "curso_col": 0,
        "turno_col": 1,
        "campus_col": None,
        "campus_prefixo": "Campus UnB — ",
        "modalidades": {
            (2,3): "EP_R1_PPI",
            (4,5): "EP_R1_PPI_PCD",
            (6,7): "EP_R1_NPPI",
            (8,9): "EP_R1_NPPI_PCD",
            (10,11): "EP_R2_PPI",
            (12,13): "EP_R2_PPI_PCD",
            (14,15): "EP_R2_NPPI",
            (16,17): "EP_R2_NPPI_PCD",
            (18,19): "CN",
            (20,21): "AC"
        }
    },

    "2021-2023": {
        "header": 4,
        "tipo": CAMPUS_LINHA,
        "curso_col": 0,
        "turno_col": 1,
        "campus_col": None,
        "campus_prefixo": "Campus UnB — ",
        "modalidades": {
            (2,3): "EP_R1_PPI",
            (4,5): "EP_R1_PPI_PCD",
            (6,7): "EP_R1_NPPI",
            (8,9): "EP_R1_NPPI_PCD",
            (10,11): "EP_R2_PPI",
            (12,13): "EP_R2_PPI_PCD",
            (14,15): "EP_R2_NPPI",
            (16,17): "EP_R2_NPPI_PCD",
            (18,19): "CN",
            (20,21): "AC"
        }
    },

    "2022-2024": {
        "header": 4,
        "tipo": CAMPUS_LINHA,
        "curso_col": 0,
        "turno_col": 1,
        "campus_col": None,
        "campus_prefixo": "Campus UnB — ",
        "modalidades": {
            (2,3): "EP_R1_PPI",
            (4,5): "EP_R1_PPI_PCD",
            (6,7): "EP_R1_NPPI",
            (8,9): "EP_R1_NPPI_PCD",
            (10,11): "EP_R2_PPI",
            (12,13): "EP_R2_PPI_PCD",
            (14,15): "EP_R2_NPPI",
            (16,17): "EP_R2_NPPI_PCD",
            (18,19): "CN",
            (20,21): "AC"
        }
    },

    "2023-2025": {
        "header": 4,
        "tipo": CAMPUS_LINHA,
        "curso_col": 0,
        "turno_col": 1,
        "campus_col": None,
        "campus_prefixo": "Campus UnB — ",
        "modalidades": {
            (2,3): "EP_R1_PPIQ",
            (4,5): "EP_R1_PPIQ_PCD",
            (6,7): "EP_R1_NPPIQ",
            (8,9): "EP_R1_NPPIQ_PCD",
            (10,11): "EP_R2_PPIQ",
            (12,13): "EP_R2_PPIQ_PCD",
            (14,15): "EP_R2_NPPIQ",
            (16,17): "EP_R2_NPPIQ_PCD",
            (18,19): "CN",
            (20,21): "AC"
        }
    }

}

print("=" * 70)
print("BIBLIOTECA CARREGADA")
print("=" * 70)

print(f"Triênios............. {len(CONFIG_TRIENIOS)}")
print(f"Modalidades.......... {len(MODALIDADES)}")
print(f"Cursos canônicos..... {len(CATALOGO_CURSOS)}")
print(f"Variantes............ {len(INDICE_CURSOS)}")

BIBLIOTECA CARREGADA
Triênios............. 6
Modalidades.......... 18
Cursos canônicos..... 90
Variantes............ 132


In [10]:
# ==========================================================
# LEITURA
# ==========================================================

def carregar_tsv(arquivo):
    """
    Carrega um TSV do PAS detectando automaticamente
    a linha do cabeçalho.
    """

    bruto = pd.read_csv(
        arquivo,
        sep="\t",
        header=None,
        dtype=str,
        keep_default_na=False
    )

    cabecalho = next(

        (
            i
            for i, linha in bruto.iterrows()
            if {"Curso", "Turno"}.issubset(
                linha.astype(str).str.strip()
            )
        ),

        None

    )

    if cabecalho is None:
        raise ValueError(
            f"Cabeçalho não encontrado em {arquivo.name}"
        )

    df = pd.read_csv(
        arquivo,
        sep="\t",
        header=cabecalho,
        dtype=str,
        keep_default_na=False
    )

    df.columns = (
        df.columns
          .astype(str)
          .str.strip()
    )

    return df

In [11]:
# ==========================================================
# LIMPEZA
# ==========================================================

def limpar_texto(valor):
    """Normaliza valores textuais."""

    if pd.isna(valor):
        return ""

    return (
        str(valor)
        .replace("\n", " ")
        .replace("\t", " ")
        .replace("*", "")
        .strip()
    )


def limpar_nota(valor):
    """Converte notas para float."""

    if pd.isna(valor):
        return np.nan

    valor = (
        str(valor)
        .strip()
        .replace(",", ".")
    )

    if valor in {"", "-"}:
        return np.nan

    try:
        return float(valor)
    except ValueError:
        return np.nan

In [12]:
# ==========================================================
# NORMALIZAÇÃO
# ==========================================================

def normalizar_curso(curso):

    curso = limpar_texto(curso)

    return INDICE_CURSOS.get(
        curso,
        curso
    )


# ==========================================================
# NORMALIZAÇÃO DOS CAMPI
# ==========================================================

def normalizar_campus(campus):
    """
    Normaliza os nomes dos campi da UnB.

    Todas as variantes conhecidas são convertidas para um dos
    quatro campi oficiais.
    """

    campus = limpar_texto(campus)

    campus = campus.replace("\u00a0", " ")

    campus = re.sub(

        r"\s+",

        " ",

        campus

    ).strip()

    # ------------------------------------------------------
    # Catálogo
    # ------------------------------------------------------

    if campus in INDICE_CAMPI:

        return INDICE_CAMPI[campus]

    # ------------------------------------------------------
    # Fallback
    # ------------------------------------------------------

    texto = campus.lower()

    if "darcy" in texto:

        return "Darcy Ribeiro"

    if "ceil" in texto:

        return "Ceilândia"

    if "gama" in texto:

        return "Gama"

    if "planalt" in texto:

        return "Planaltina"

    return campus


def registrar_modalidade(
    modalidade,
    trienio
):
    """
    Retorna os metadados de uma modalidade.
    """

    info = MODALIDADES[
        modalidade
    ].copy()

    if info["faixa_renda"]:

        info["limite_salario_minimo"] = (
            CONFIG_TRIENIOS[
                trienio
            ]["limite_r1"]
        )

    else:

        info["limite_salario_minimo"] = None

    return info

In [13]:
[k for k in CATALOGO_CURSOS.keys() if "Música" in k]

['Música (Bacharelado)', 'Música (Licenciatura)']

In [14]:
# ==========================================================
# LEITURA
# ==========================================================

def carregar_tsv(arquivo, header):

    return (
        pd.read_csv(
            arquivo,
            sep="\t",
            header=None,
            dtype=str
        )
        .iloc[header:]
        .reset_index(drop=True)
    )

In [15]:
# ==========================================================
# PARSER
# ==========================================================

def parse_pas(trienio, arquivo):

    cfg = CONFIG_TRIENIOS[trienio]

    layout = LAYOUTS[trienio]

    dados = carregar_tsv(
        arquivo,
        layout["header"]
    )

    registros = []

    campus = None

    for _, row in dados.iterrows():

        # -----------------------------
        # Campus
        # -----------------------------

        if layout["tipo"] == CAMPUS_LINHA:

            texto = limpar_texto(
                row.iloc[0]
            )

            if texto.startswith(
                layout["campus_prefixo"]
            ):

                campus = normalizar_campus(
                    texto.replace(
                        layout["campus_prefixo"],
                        ""
                    )
                )

                continue

        else:

            campus = normalizar_campus(
                row.iloc[
                    layout["campus_col"]
                ]
            )

        curso = normalizar_curso(
            row.iloc[
                layout["curso_col"]
            ]
        )

        turno = limpar_texto(
            row.iloc[
                layout["turno_col"]
            ]
        )

        if not all([campus, curso, turno]):
            continue

        for (cmin, cmax), modalidade in layout["modalidades"].items():

            nota_min = limpar_nota(
                row.iloc[cmin]
            )

            nota_max = limpar_nota(
                row.iloc[cmax]
            )

            if (
                pd.isna(nota_min)
                and
                pd.isna(nota_max)
            ):
                continue

            registros.append({

                "Subprograma": trienio,

                "Ano": cfg["ano_ingresso"],

                "Campus": campus,

                "Curso": curso,

                "Turno": turno,

                "Modalidade": modalidade,

                **registrar_modalidade(
                    modalidade,
                    trienio
                ),

                "Nota Mínima": nota_min,

                "Nota Máxima": nota_max

            })

    return pd.DataFrame(registros)

In [16]:
# ==========================================================
# PARSER
# ==========================================================

def parse_pas(trienio, arquivo):

    cfg = CONFIG_TRIENIOS[trienio]

    layout = LAYOUTS[trienio]

    dados = carregar_tsv(
        arquivo,
        layout["header"]
    )

    registros = []

    campus = None

    for _, row in dados.iterrows():

        # -----------------------------
        # Campus
        # -----------------------------

        if layout["tipo"] == CAMPUS_LINHA:

            texto = limpar_texto(
                row.iloc[0]
            )

            if texto.startswith(
                layout["campus_prefixo"]
            ):

                campus = normalizar_campus(
                    texto.replace(
                        layout["campus_prefixo"],
                        ""
                    )
                )

                continue

        else:

            campus = normalizar_campus(
                row.iloc[
                    layout["campus_col"]
                ]
            )

        curso = normalizar_curso(
            row.iloc[
                layout["curso_col"]
            ]
        )

        turno = limpar_texto(
            row.iloc[
                layout["turno_col"]
            ]
        )

        if not all([campus, curso, turno]):
            continue

        for (cmin, cmax), modalidade in layout["modalidades"].items():

            nota_min = limpar_nota(
                row.iloc[cmin]
            )

            nota_max = limpar_nota(
                row.iloc[cmax]
            )

            if (
                pd.isna(nota_min)
                and
                pd.isna(nota_max)
            ):
                continue

            registros.append({

                "Subprograma": trienio,

                "Ano": cfg["ano_ingresso"],

                "Campus": campus,

                "Curso": curso,

                "Turno": turno,

                "Modalidade": modalidade,

                **registrar_modalidade(
                    modalidade,
                    trienio
                ),

                "Nota Mínima": nota_min,

                "Nota Máxima": nota_max

            })

    return pd.DataFrame(registros)

In [17]:
# ==========================================================
# VALIDAÇÃO
# ==========================================================

def validar(df):
    """
    Valida a consistência do DataFrame normalizado.
    """

    obrigatorias = [
        "Campus",
        "Curso",
        "Turno",
        "Modalidade",
        "Nota Mínima",
        "Nota Máxima",
    ]

    faltantes = [
        c
        for c in obrigatorias
        if c not in df.columns
    ]

    if faltantes:
        raise ValueError(
            f"Colunas obrigatórias ausentes: {faltantes}"
        )

    if df.empty:
        raise ValueError(
            "Nenhum registro foi processado."
        )

    chave = [
        "Campus",
        "Curso",
        "Turno",
        "Modalidade",
    ]

    duplicados = df.duplicated(
        subset=chave,
        keep=False
    )

    if duplicados.any():

        print("=" * 70)
        print("REGISTROS DUPLICADOS")
        print("=" * 70)

        display(
            df.loc[duplicados]
            .sort_values(chave)
        )

        raise ValueError(
            "Existem registros duplicados."
        )

    return df

# ==========================================================
# DIAGNÓSTICO
# ==========================================================

def diagnostico(df):

    print("=" * 70)
    print("DIAGNÓSTICO")
    print("=" * 70)

    print(f"Registros..... {len(df):,}")
    print(f"Campi......... {df['Campus'].nunique()}")
    print(f"Cursos........ {df['Curso'].nunique()}")
    print(f"Turnos........ {df['Turno'].nunique()}")
    print(f"Modalidades... {df['Modalidade'].nunique()}")

    print()

    display(df.head())

In [18]:
# ==========================================================
# EXPORTAÇÃO
# ==========================================================

def salvar_resultados(
    df,
    trienio
):

    csv = (
        DIR_SAIDA /
        f"{trienio}.csv"
    )

    json_ = (
        DIR_SAIDA /
        f"{trienio}.json"
    )

    df.to_csv(
        csv,
        index=False,
        encoding="utf-8-sig"
    )

    df.to_json(
        json_,
        orient="records",
        indent=4,
        force_ascii=False
    )

    print()

    print("Arquivos gerados:")

    print(f" • {csv.name}")

    print(f" • {json_.name}")

In [19]:
# ==========================================================
# PIPELINE
# ==========================================================

def executar(trienio):
    """
    Executa todo o processamento de um boletim do PAS.
    """

    arquivo = DIR_TSV / f"{trienio}.tsv"

    print("=" * 70)
    print(trienio)
    print("=" * 70)

    df = parse_pas(
        trienio,
        arquivo
    )

    validar(df)

    diagnostico(df)

    salvar_resultados(
        df,
        trienio
    )

    return df